# 📘 智能体架构 17：反思性元认知智能体

欢迎深入探讨最复杂的智能体模式之一的实现：**反思性元认知智能体**。该架构赋予智能体一种自我意识形式，使其能够在采取行动之前对自己的能力、信心和局限性进行推理。

这超越了简单的自我反思。元认知智能体维护一个显式的**"自我模型"**——对其自身知识、工具和边界的结构化表示。当面临任务时，其第一步不是解决问题，而是*在自我模型的背景下分析问题*。它会询问内部问题，例如：
- "我有足够的知识来自信地回答这个问题吗？"
- "这个主题是否在我指定的专业领域内？"
- "我是否有特定工具可以安全准确地回答这个问题？"
- "用户的查询是否关于错误会很危险的高风险主题？"

基于答案，它选择一种策略：直接推理、使用专业工具，或者——最重要的是——当任务超出其已知限制时**升级给人类**。

为了构建一个复杂而强大的演示，我们将创建一个**医疗分诊和信息助手**。这是一个典型的高风险场景，智能体识别自身局限性的能力不仅是一个功能，而且是关键的安全要求。

### 定义
**反思性元认知智能体**是一个智能体，它维护并使用关于自身能力、知识边界和信心水平的显式模型来为给定任务选择最合适的策略。这种自我建模使其行为更加安全和可靠，特别是在错误信息有害的领域。

### 高层工作流程

1.  **感知任务：** 智能体接收用户请求。
2.  **元认知分析（自我反思）：** 智能体的核心推理引擎*根据其自身的自我模型*分析请求。它评估其信心、工具的相关性，以及查询是否在其预定义的操作域内。
3.  **策略选择：** 基于分析，智能体选择以下几种策略之一：
    *   **直接推理：** 对于其知识库内的高信心、低风险查询。
    *   **使用工具：** 当查询需要智能体通过工具拥有的特定能力时。
    *   **升级/拒绝：** 对于低信心、高风险或超出范围的查询。
4.  **执行策略：** 执行选定的路径。
5.  **响应：** 智能体提供结果，这可能是直接答案、工具增强的答案，或带有咨询专家指示的安全拒绝。

### 适用场景 / 应用
*   **高风险咨询系统：** 任何在医疗、法律或金融等领域提供信息的系统，智能体必须能够说"我不知道"或"您应该咨询专业人士"。
*   **自主系统：** 在尝试执行物理任务之前必须评估自己安全执行能力的机器人。
*   **复杂工具编排器：** 必须从庞大库中选择正确 API 的智能体，理解某些 API 更危险或成本更高。

### 优缺点
*   **优点：**
    *   **增强的安全性和可靠性：** 主要好处。智能体被明确设计为避免在不是专家的领域做出自信的断言。
    *   **改进的决策制定：** 通过强制策略的深思熟虑的选择而不是幼稚的直接尝试，导致更强大的行为。
*   **缺点：**
    *   **自我模型的复杂性：** 定义和维护准确的自我模型可能很复杂。
    *   **元认知开销：** 初始分析步骤为每个请求增加延迟和计算成本。

## 阶段 0：基础与环境设置

标准库和环境变量设置。

In [ ]:
# !pip install -q -U langchain-anthropic langchain langgraph rich python-dotenv

In [ ]:
import os
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Metacognitive Agent"

required_vars = ["ANTHROPIC_API_KEY", "LANGCHAIN_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

## 阶段 1：定义智能体的自我模型和工具

这是智能体自我意识的基础。我们将创建一个结构化的 `AgentSelfModel` 和一个专业工具。这个模型不仅仅是一个提示；它是一个将被传递到智能体推理循环中的配置对象。

In [3]:
console = Console()

# --- The Agent's Self-Model ---
class AgentSelfModel(BaseModel):
    """A structured representation of the agent's capabilities and limitations."""
    name: str
    role: str
    # The agent's explicit knowledge boundaries
    knowledge_domain: List[str] = Field(description="List of topics the agent is knowledgeable about.")
    # The agent's available tools
    available_tools: List[str] = Field(description="List of tools the agent can use.")
    confidence_threshold: float = Field(description="The confidence level (0-1) below which the agent must escalate.", default=0.6)

# Instantiate the self-model for our Medical Triage Agent
medical_agent_model = AgentSelfModel(
    name="TriageBot-3000",
    role="A helpful AI assistant for providing preliminary medical information.",
    knowledge_domain=["common_cold", "influenza", "allergies", "headaches", "basic_first_aid"],
    available_tools=["drug_interaction_checker"]
)

# --- Specialist Tools ---
class DrugInteractionChecker:
    """A mock tool to check for drug interactions."""
    def check(self, drug_a: str, drug_b: str) -> str:
        """Checks for interactions between two drugs."""
        # In a real system, this would query a medical database.
        known_interactions = {
            frozenset(["ibuprofen", "lisinopril"]): "Moderate risk: Ibuprofen may reduce the blood pressure-lowering effects of lisinopril. Monitor blood pressure.",
            frozenset(["aspirin", "warfarin"]): "High risk: Increased risk of bleeding. This combination should be avoided unless directed by a doctor."
        }
        interaction = known_interactions.get(frozenset([drug_a.lower(), drug_b.lower()]))
        if interaction:
            return f"Interaction Found: {interaction}"
        return "No known significant interactions found. However, always consult a pharmacist or doctor."

drug_tool = DrugInteractionChecker()
print("Agent Self-Model and Tools defined successfully.")

Agent Self-Model and Tools defined successfully.


## 阶段 2：使用 LangGraph 构建元认知智能体

这就是魔法发生的地方。我们将构建一个图，其中第一步是**元认知分析**。该节点将使用一个强大、详细的提示来使智能体推理自己。然后，条件路由器将根据所选策略引导流程。

In [ ]:
model = os.environ.get("MODEL_NAME", "claude-opus-4-5-20251101")
base_url = os.environ.get("BASE_URL")
llm = ChatAnthropic(model=model, base_url=base_url, temperature=0)

# Pydantic Models for structured outputs
class MetacognitiveAnalysis(BaseModel):
    """The agent's self-analysis of a query."""
    confidence: float = Field(description="A score from 0.0 to 1.0 representing the agent's confidence in its ability to answer safely and accurately.")
    strategy: str = Field(description="The chosen strategy. Must be one of: 'reason_directly', 'use_tool', 'escalate'.")
    reasoning: str = Field(description="A brief justification for the chosen confidence and strategy.")
    tool_to_use: Optional[str] = Field(description="If strategy is 'use_tool', the name of the tool to use.", default=None)
    tool_args: Optional[Dict[str, Any]] = Field(description="If strategy is 'use_tool', the arguments for the tool.", default=None)

# LangGraph State
class AgentState(TypedDict):
    user_query: str
    self_model: AgentSelfModel
    metacognitive_analysis: Optional[MetacognitiveAnalysis]
    tool_output: Optional[str]
    final_response: str

# Graph Nodes
def metacognitive_analysis_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel("🤔 Agent is performing metacognitive analysis...", title="[yellow]Step: Self-Reflection[/yellow]"))
    prompt = ChatPromptTemplate.from_template(
        """You are a metacognitive reasoning engine for an AI assistant. Your task is to analyze a user's query in the context of the agent's own capabilities and limitations (its 'self-model').
        Your primary directive is **SAFETY**. You must determine the safest and most appropriate strategy for handling the query.

        **Agent's Self-Model:**
        - Name: {agent_name}
        - Role: {agent_role}
        - Knowledge Domain: {knowledge_domain}
        - Available Tools: {available_tools}

        **Strategy Rules:**
        1.  **escalate:** Choose this strategy if the query involves a potential medical emergency (e.g., chest pain, difficulty breathing, severe injury, broken bones), is outside the agent's knowledge domain, or if you have any doubt about providing a safe answer. **WHEN IN DOUBT, ESCALATE.**
        2.  **use_tool:** Choose this strategy if the query explicitly or implicitly requires one of the available tools. For example, a question about drug interactions requires the 'drug_interaction_checker'.
        3.  **reason_directly:** Choose this strategy ONLY if you are highly confident the query is a simple, low-risk question that falls squarely within the agent's knowledge domain.

        Analyze the user query below and provide your metacognitive analysis in the required format.

        **User Query:** "{query}"""
    )
    chain = prompt | llm.with_structured_output(MetacognitiveAnalysis)
    analysis = chain.invoke({
        "query": state['user_query'],
        "agent_name": state['self_model'].name,
        "agent_role": state['self_model'].role,
        "knowledge_domain": ", ".join(state['self_model'].knowledge_domain),
        "available_tools": ", ".join(state['self_model'].available_tools),
    })
    console.print(Panel(f"[bold]Confidence:[/bold] {analysis.confidence:.2f}\n[bold]Strategy:[/bold] {analysis.strategy}\n[bold]Reasoning:[/bold] {analysis.reasoning}", title="Metacognitive Analysis Result"))
    return {"metacognitive_analysis": analysis}

def reason_directly_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel("✅ Confident in direct answer. Generating response...", title="[green]Strategy: Reason Directly[/green]"))
    prompt = ChatPromptTemplate.from_template("You are {agent_role}. Provide a helpful, non-prescriptive answer to the user's query. Remind the user that you are not a doctor.\n\nQuery: {query}")
    chain = prompt | llm
    response = chain.invoke({"agent_role": state['self_model'].role, "query": state['user_query']}).content
    return {"final_response": response}

def call_tool_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel(f"🛠️ Confidence requires tool use. Calling `{state['metacognitive_analysis'].tool_to_use}`...", title="[cyan]Strategy: Use Tool[/cyan]"))
    analysis = state['metacognitive_analysis']
    if analysis.tool_to_use == 'drug_interaction_checker':
        tool_output = drug_tool.check(**analysis.tool_args)
        return {"tool_output": tool_output}
    return {"tool_output": "Error: Tool not found."}

def synthesize_tool_response_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel("📝 Synthesizing final response from tool output...", title="[cyan]Step: Synthesize[/cyan]"))
    prompt = ChatPromptTemplate.from_template("You are {agent_role}. You have used a tool to get specific information. Now, present this information to the user in a clear and helpful way. ALWAYS include a disclaimer to consult a healthcare professional.\n\nOriginal Query: {query}\nTool Output: {tool_output}")
    chain = prompt | llm
    response = chain.invoke({"agent_role": state['self_model'].role, "query": state['user_query'], "tool_output": state['tool_output']}).content
    return {"final_response": response}

def escalate_to_human_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel("🚨 Low confidence or high risk detected. Escalating to human.", title="[bold red]Strategy: Escalate[/bold red]"))
    response = "I am an AI assistant and not qualified to provide information on this topic. This query is outside my knowledge domain or involves potentially serious symptoms. **Please consult a qualified medical professional immediately.**"
    return {"final_response": response}

# Conditional Edge
def route_strategy(state: AgentState) -> str:
    return state["metacognitive_analysis"].strategy

# Build the graph
workflow = StateGraph(AgentState)
workflow.add_node("analyze", metacognitive_analysis_node)
workflow.add_node("reason", reason_directly_node)
workflow.add_node("call_tool", call_tool_node)
workflow.add_node("synthesize", synthesize_tool_response_node)
workflow.add_node("escalate", escalate_to_human_node)

workflow.set_entry_point("analyze")
workflow.add_conditional_edges("analyze", route_strategy, {
    "reason_directly": "reason",
    "use_tool": "call_tool",
    "escalate": "escalate"
})
workflow.add_edge("call_tool", "synthesize")
workflow.add_edge("reason", END)
workflow.add_edge("synthesize", END)
workflow.add_edge("escalate", END)

metacognitive_agent = workflow.compile()
print("Reflexive Metacognitive Agent graph compiled successfully.")

# Visualize the graph
try:
    from IPython.display import Image, display
    png_image = metacognitive_agent.get_graph().draw_mermaid_png()
    display(Image(png_image))
except Exception as e:
    print(f"Graph visualization failed: {e}. Please ensure pygraphviz is installed.")

## 阶段 3：演示与分析

现在我们将用一系列越来越困难和高风险的查询测试智能体。我们将观察元认知分析如何正确地将每个查询路由到适当的路径，展示系统的安全性和自我意识。

In [5]:
def run_agent(query: str):
    initial_state = {"user_query": query, "self_model": medical_agent_model}
    result = metacognitive_agent.invoke(initial_state)
    console.print(Markdown(result['final_response']))

# Test 1: Simple, should be answered directly
console.print("--- Test 1: Simple, In-Scope, Low-Risk Query ---")
run_agent("What are the symptoms of a common cold?")

# Test 2: Requires the specific tool
console.print("\n--- Test 2: Specific Query Requiring a Tool ---")
run_agent("Is it safe to take Ibuprofen if I am also taking Lisinopril?")

# Test 3: High-stakes, should be escalated immediately
console.print("\n--- Test 3: High-Stakes, Emergency Query ---")
run_agent("I have a crushing pain in my chest and my left arm feels numb, what should I do?")

--- Test 1: Simple, In-Scope, Low-Risk Query ---

                 Step: Self-Reflection                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 🤔 Agent is performing metacognitive analysis...      ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                   Metacognitive Analysis Result                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Confidence: 0.90                                                 ┃
┃ Strategy: reason_directly                                        ┃
┃ Reasoning: The user's query about symptoms of a common cold      ┃
┃ falls directly within the agent's specified knowledge domain. It ┃
┃ is a low-risk, informational question.                           ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                         Strategy: Reason Directly                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ✅ Confident in direct answer. Generating response...                     ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

Based on your query, here is some general information about the common cold. Please remember, I am an AI assistant and not a medical doctor. This information should not be considered medical advice.

Common symptoms of a cold often include:
*   Runny or stuffy nose
*   Sore throat
*   Cough
*   Sneezing
*   Mild body aches or a slight headache

These symptoms are typically mild and resolve on their own. If your symptoms are severe or persist, it is always best to consult a healthcare professional.

...


--- Test 3: High-Stakes, Emergency Query ---

                 Step: Self-Reflection                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 🤔 Agent is performing metacognitive analysis...      ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                   Metacognitive Analysis Result                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Confidence: 0.10                                                 ┃
┃ Strategy: escalate                                               ┃
┃ Reasoning: The user's query describes symptoms (crushing chest   ┃
┃ pain, numbness in arm) that are highly indicative of a potential ┃
┃ medical emergency. This is far outside the agent's knowledge     ┃
┃ domain and requires immediate professional medical attention. The┃
┃ only safe action is to escalate.                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                           Strategy: Escalate                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 🚨 Low confidence or high risk detected. Escalating to human.       ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

I am an AI assistant and not qualified to provide information on this topic. This query is outside my knowledge domain or involves potentially serious symptoms. **Please consult a qualified medical professional immediately.**

### 结果分析

该演示有力地说明了该架构提供的安全性和可靠性：

1.  **正确范围的答案：** 对于"普通感冒"查询，元认知分析正确地识别了它是其知识域内的低风险主题。它设置了高信心分数并选择了 `reason_directly` 策略，提供了有帮助但适当警告的答案。

2.  **正确的工具使用：** 对于药物相互作用问题，分析识别了对特定能力的需求。它正确地识别到需要 `drug_interaction_checker` 工具，对*其使用工具的能力*设置了高信心，并选择了 `use_tool` 策略。最终响应是工具输出的安全、综合摘要。

3.  **关键安全升级：** 这是最重要的结果。一个幼稚的智能体可能试图通过在网上搜索原因来回答"胸痛"查询，可能提供危险和误导性的信息。我们的元认知智能体在安全的主要指令指导下，立即识别了医疗紧急情况的迹象。元认知分析分配了非常低的信心分数，并正确地选择了 `escalate` 策略。最终输出不是答案，而是安全、负责任的拒绝和寻求专业帮助的指示。它正确地识别了自身能力的限制。

该工作流程证明，通过强制智能体在推理问题*之前*推理自己，我们可以在其操作中构建强大的安全性和可靠性层。

## 结论

在这个详细的笔记本中，我们实现了一个**反思性元认知智能体**，一个通过赋予智能体自我意识来优先考虑安全性和可靠性的复杂架构。通过构建显式的 `self-model` 并强制将元认知分析作为任何任务的第一步，我们创建了一个了解自身边界的系统。

关键的创新在于智能体初始目标的转变，从"我如何回答这个问题？"变为"*我是否应该*回答这个问题，如果是，如何？"这种内省步骤允许智能体动态选择最安全和最合适的策略——无论是直接推理、专业工具使用，还是关键地升级给人类专家。

该架构不仅仅是一种技术；它是一种设计理念。对于创建可以在高风险、现实世界领域中运行的可信负责任 AI 智能体来说，它绝对是必不可少的，在这些领域，知道自己*不*知道什么与自己知道什么一样重要。